In [1]:
# 1. Cài đặt thư viện cần thiết
!pip install py-vncorenlp transformers sentencepiece


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.4 MB/s eta 0:00:00
  Created wheel for py-vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=4514c29f5db664df33d499e8273dd6a9afc0f6a47362940e23ee2f3862860891
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py-vncorenlp


In [ ]:
import os
import shutil
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, T5EncoderModel
from tqdm.auto import tqdm
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from torch.cuda.amp import autocast, GradScaler
import py_vncorenlp


class Config:
    TRAIN_PATH = '/kaggle/input/neu-esc-dataset/train_set.csv'
    VAL_PATH = '/kaggle/input/neu-esc-dataset/val_set.csv'
    
    # Đổi sang ViT5 Base
    MODEL_NAME = "VietAI/vit5-base"
    MAX_LEN = 512           
    BATCH_SIZE = 8          
    ACCUMULATION_STEPS = 4  
    
    EPOCHS = 20
    LEARNING_RATE = 3e-5
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    RDR_PATH = '/kaggle/working/vncorenlp_fixed'


2026-01-12 19:24:30.591070: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768245870.770198      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768245870.819994      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768245871.241833      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768245871.241872      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768245871.241875      24 computation_placer.cc:177] computation placer alr

In [3]:
def setup_vncorenlp():
    should_download = True
    if os.path.exists(Config.RDR_PATH):
        if os.path.exists(os.path.join(Config.RDR_PATH, 'VnCoreNLP-1.2.jar')):
            should_download = False
        else:
            shutil.rmtree(Config.RDR_PATH)
    
    if should_download:
        print("Đang tải VnCoreNLP (v1.2)...")
        try:
            os.makedirs(Config.RDR_PATH, exist_ok=True)
            py_vncorenlp.download_model(save_dir=Config.RDR_PATH)
            print("✅ Tải thành công!")
        except Exception as e:
            print(f"❌ Lỗi tải VnCoreNLP: {e}")

setup_vncorenlp()

try:
    if 'rdrsegmenter' not in globals():
        rdrsegmenter = py_vncorenlp.VnCoreNLP(save_dir=Config.RDR_PATH, annotators=["wseg"])
        print("✅ RDRSegmenter đã sẵn sàng.")
    else:
        print("ℹ️ RDRSegmenter đang chạy.")
except Exception as e:
    print(f"⚠️ Lỗi khởi tạo Java: {e}")

def segment_text(text):
    if not isinstance(text, str) or not text.strip(): return ""
    try:
        sentences = rdrsegmenter.word_segment(text)
        return " ".join(sentences)
    except:
        return text


Đang tải VnCoreNLP (v1.2)...


--2026-01-12 19:24:43--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 7.12M 4s
    50K .......... .......... .......... .......... ..........  0% 8.34M 3s
   100K .......... .......... .......... .......... ..........  0% 82.3M 2s
   150K .......... .......... .......... .......... ..........  0% 30.6M 2s
   200K .......... .......... .......... .......... ..........  0% 55.1M 2s
   250K .......... .......... .......... .......... ..........  1% 12.6M 2s
   300K .......... .......... .......... .......... ..........  1%  232M 2s
   350K ..

✅ Tải thành công!
2026-01-12 19:24:50 INFO  WordSegmenter:24 - Loading Word Segmentation model
✅ RDRSegmenter đã sẵn sàng.


In [4]:
class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = df['text'].values
        self.sentiments = df['sentiment'].values
        self.topics = df['classification'].values 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        raw_text = str(self.texts[index])
        text = segment_text(raw_text) # ViT5 vẫn hưởng lợi từ word segmentation
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'sentiment_label': torch.tensor(self.sentiments[index], dtype=torch.long),
            'topic_label': torch.tensor(self.topics[index], dtype=torch.long)
        }

In [ ]:
class MultiTaskViT5(nn.Module):
    def __init__(self, n_sentiment, n_topic):
        super(MultiTaskViT5, self).__init__()
        print(f"Loading {Config.MODEL_NAME}...")
        
        self.t5_encoder = T5EncoderModel.from_pretrained(Config.MODEL_NAME)
        
        self.t5_encoder.gradient_checkpointing_enable() 
        
        hidden_size = self.t5_encoder.config.d_model
        self.drop = nn.Dropout(p=0.1)
        
        self.sent_out = nn.Linear(hidden_size, n_sentiment)
        self.topic_out = nn.Linear(hidden_size, n_topic)

    def forward(self, input_ids, attention_mask):
        # T5 Encoder trả về last_hidden_state: [Batch, Seq_Len, Hidden]
        outputs = self.t5_encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        output = self.drop(mean_pooled)
        
        return self.sent_out(output), self.topic_out(output)


In [ ]:
def get_class_weights(y, device):
    class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return torch.tensor(class_weights, dtype=torch.float).to(device)

def train_epoch(model, dataloader, optimizer, device, scaler, w_sent, w_topic):
    model.train()
    total_loss = 0
    
    crit_sent  = nn.CrossEntropyLoss(weight=w_sent, label_smoothing=0.05)
    crit_topic = nn.CrossEntropyLoss(weight=w_topic, label_smoothing=0.1)
    
    optimizer.zero_grad()
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        sent_lbl = batch['sentiment_label'].to(device)
        topic_lbl = batch['topic_label'].to(device)
        with autocast():
            sent_logits, topic_logits = model(input_ids, mask)
            loss_s = crit_sent(sent_logits, sent_lbl)
            loss_t = crit_topic(topic_logits, topic_lbl)
            loss = (loss_s + loss_t) / Config.ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        
        if (step + 1) % Config.ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        current_loss = loss.item() * Config.ACCUMULATION_STEPS
        total_loss += current_loss
        del input_ids, mask, sent_lbl, topic_lbl, sent_logits, topic_logits, loss
        
        progress_bar.set_postfix({'loss': f"{current_loss:.4f}"})
        
    return total_loss / len(dataloader)

def validate_epoch(model, dataloader, device, w_sent, w_topic):
    model.eval()
    total_loss = 0
    correct_sent = 0
    correct_topic = 0
    total_samples = 0
    
    crit_sent  = nn.CrossEntropyLoss(weight=w_sent, label_smoothing=0.05)
    crit_topic = nn.CrossEntropyLoss(weight=w_topic, label_smoothing=0.1)
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            sent_lbl = batch['sentiment_label'].to(device)
            topic_lbl = batch['topic_label'].to(device)

            sent_logits, topic_logits = model(input_ids, mask)
            
            loss = crit_sent(sent_logits, sent_lbl) + crit_topic(topic_logits, topic_lbl)
            total_loss += loss.item()
            
            pred_sent = torch.argmax(sent_logits, dim=1)
            correct_sent += (pred_sent == sent_lbl).sum().item()
            
            pred_topic = torch.argmax(topic_logits, dim=1)
            correct_topic += (pred_topic == topic_lbl).sum().item()
            
            total_samples += sent_lbl.size(0)
            
    return total_loss / len(dataloader), correct_sent / total_samples, correct_topic / total_samples


In [7]:
if __name__ == '__main__':
    print("--- 1. Loading Data ---")
    try:
        df_train = pd.read_csv(Config.TRAIN_PATH)
        df_val = pd.read_csv(Config.VAL_PATH)
        print(f"Train: {len(df_train)} | Val: {len(df_val)}")
    except:
        print("⚠️ Dùng dummy data để test code")
        df_train = pd.DataFrame({'text': ['Test'], 'sentiment': [0], 'classification': [0]})
        df_val = df_train.copy()

    # Xác định số class
    n_sent = int(df_train['sentiment'].max()) + 1
    n_topic = int(df_train['classification'].max()) + 1
    print(f"Classes: {n_sent} Sentiments, {n_topic} Topics")

    print("\n--- 2. Tokenizer & Dataset ---")
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
    
    train_ds = MultiTaskDataset(df_train, tokenizer, Config.MAX_LEN)
    val_ds = MultiTaskDataset(df_val, tokenizer, Config.MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, num_workers=0)
    
    # Class Weights
    w_sent = get_class_weights(df_train['sentiment'].values, Config.DEVICE)
    w_topic = get_class_weights(df_train['classification'].values, Config.DEVICE)

    print("\n--- 3. Initializing ViT5 ---")
    model = MultiTaskViT5(n_sent, n_topic)
    model.to(Config.DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    scaler = GradScaler() # Cho Mixed Precision

    print("\n--- 4. Start Training ---")
    best_val_loss = float('inf')
    
    for epoch in range(Config.EPOCHS):
        print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
        
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, Config.DEVICE, scaler, w_sent, w_topic)
        
        # Valid
        val_loss, val_acc_s, val_acc_t = validate_epoch(model, val_loader, Config.DEVICE, w_sent, w_topic)
        
        print(f" -> Train Loss: {train_loss:.4f}")
        print(f" -> Val Loss  : {val_loss:.4f} | Sent Acc: {val_acc_s:.4f} | Topic Acc: {val_acc_t:.4f}")
        
        # Save Best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "vit5_best.pth")
            print(" ✅ Saved Best Model (based on Val Loss)")
            
    print("\n✅ Training Completed!")

--- 1. Loading Data ---
Train: 23048 | Val: 3305
Classes: 4 Sentiments, 10 Topics

--- 2. Tokenizer & Dataset ---


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Validating:   0%|          | 0/414 [00:00<?, ?it/s]

 -> Train Loss: 2.3383
 -> Val Loss  : 4.0502 | Sent Acc: 0.8097 | Topic Acc: 0.7522

Epoch 20/20


Training:   0%|          | 0/2881 [00:00<?, ?it/s]

Validating:   0%|          | 0/414 [00:00<?, ?it/s]

 -> Train Loss: 2.3132
 -> Val Loss  : 3.9951 | Sent Acc: 0.8175 | Topic Acc: 0.7761

✅ Training Completed!


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import torch
from tqdm.auto import tqdm

ID2SENTIMENT = {
    0: "Neutral", 1: "Positive", 2: "Negative", 3: "Toxic"
}
ID2TOPIC = {
    0: "Spam", 1: "News", 2: "Academic", 3: "Other", 4: "Service",
    5: "Jobs & Recruitment", 6: "Personal Affairs", 7: "Social Affairs",
    8: "Help & Share", 9: "Club & Events"
}


print("--- 1. Loading & Processing Test Data ---")
TEST_PATH = '/kaggle/input/neu-esc-dataset/test_set.csv'

try:
    df_test = pd.read_csv(TEST_PATH)
    print(f"Test size raw: {len(df_test)}")
    print("⏳ Đang tách từ cho tập Test...")
    tqdm.pandas()
    df_test['text'] = df_test['text'].progress_apply(lambda x: segment_text(str(x)))
    
except Exception as e:
    print(f"❌ Lỗi load file test: {e}")
    df_test = pd.DataFrame({
        'text': ['Trường này học phí đắt quá', 'Thầy cô rất nhiệt tình'],
        'sentiment': [2, 1],
        'classification': [4, 2]
    })
    df_test['text'] = df_test['text'].apply(lambda x: segment_text(x))

# Tạo Dataset & Loader
test_ds = MultiTaskDataset(df_test, tokenizer, Config.MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0)

print("\n--- 2. Loading Best Model ---")
model_best = MultiTaskViT5(n_sent, n_topic) 
model_best.to(Config.DEVICE)

try:
    model_best.load_state_dict(torch.load("vit5_best.pth"))
    print("✅ Đã load weight từ 'vit5_best.pth'")
except Exception as e:
    print(f"⚠️ Không tìm thấy file weight, dùng model hiện tại: {e}")

def evaluate_test(model, dataloader, device):
    model.eval()
    true_sents, pred_sents = [], []
    true_topics, pred_topics = [], []
    
    print("Đang chạy đánh giá trên tập Test...")
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            
            labels_s = batch['sentiment_label'].cpu().numpy()
            labels_t = batch['topic_label'].cpu().numpy()
            
            # Forward
            logits_s, logits_t = model(input_ids, mask)
            
            # Predict
            preds_s = torch.argmax(logits_s, dim=1).cpu().numpy()
            preds_t = torch.argmax(logits_t, dim=1).cpu().numpy()
            
            true_sents.extend(labels_s)
            pred_sents.extend(preds_s)
            true_topics.extend(labels_t)
            pred_topics.extend(preds_t)
            
    return true_sents, pred_sents, true_topics, pred_topics

y_true_s, y_pred_s, y_true_t, y_pred_t = evaluate_test(model_best, test_loader, Config.DEVICE)

# --- SENTIMENT REPORT ---
print("\n" + "="*40)
print("KẾT QUẢ: CẢM XÚC (SENTIMENT)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_s, y_pred_s):.4f}\n")
s_names = [ID2SENTIMENT[i] for i in sorted(ID2SENTIMENT.keys()) if i < n_sent]
print(classification_report(y_true_s, y_pred_s, target_names=s_names, zero_division=0))

# --- TOPIC REPORT ---
print("\n" + "="*40)
print("KẾT QUẢ: CHỦ ĐỀ (TOPIC)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_t, y_pred_t):.4f}\n")
t_names = [ID2TOPIC[i] for i in sorted(ID2TOPIC.keys()) if i < n_topic]
print(classification_report(y_true_t, y_pred_t, target_names=t_names, zero_division=0))

def predict_single(text, model, tokenizer, device):
    model.eval()
    # 1. Tách từ
    text_seg = segment_text(text)
    # 2. Tokenize
    encoding = tokenizer.encode_plus(
        text_seg,
        max_length=Config.MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        l_s, l_t = model(input_ids, mask)
        p_s = torch.argmax(l_s, dim=1).item()
        p_t = torch.argmax(l_t, dim=1).item()
        
    return ID2SENTIMENT.get(p_s, "Unknown"), ID2TOPIC.get(p_t, "Unknown")

print("\n--- DEMO PREDICTION ---")
sample_txt = "Trường mình cơ sở vật chất tốt nhưng học phí hơi cao so với mặt bằng chung."
print(f"Input: {sample_txt}")
s, t = predict_single(sample_txt, model_best, tokenizer, Config.DEVICE)
print(f"Sentiment: {s}")
print(f"Topic: {t}")

--- 1. Loading & Processing Test Data ---
Test size raw: 6613
⏳ Đang tách từ cho tập Test...


  0%|          | 0/6613 [00:00<?, ?it/s]


--- 2. Loading Best Model ---
Loading VietAI/vit5-base...
✅ Đã load weight từ 'vit5_best.pth'
Đang chạy đánh giá trên tập Test...


  0%|          | 0/827 [00:00<?, ?it/s]


KẾT QUẢ: CẢM XÚC (SENTIMENT)
Accuracy: 0.7479

              precision    recall  f1-score   support

     Neutral       0.93      0.73      0.81      4558
    Positive       0.49      0.78      0.61       834
    Negative       0.55      0.81      0.66      1046
       Toxic       0.78      0.81      0.79       175

    accuracy                           0.75      6613
   macro avg       0.69      0.78      0.72      6613
weighted avg       0.81      0.75      0.76      6613


KẾT QUẢ: CHỦ ĐỀ (TOPIC)
Accuracy: 0.6808

                    precision    recall  f1-score   support

              Spam       0.32      0.60      0.41        84
              News       0.35      0.61      0.44       183
          Academic       0.87      0.70      0.77      2104
             Other       0.89      0.68      0.77      2882
           Service       0.54      0.69      0.60       474
Jobs & Recruitment       0.40      0.77      0.53       163
  Personal Affairs       0.33      0.70      0.45    

In [9]:
torch.save(model.state_dict(), "vit5_last.pth")
print(" ✅ Saved Best Model (based on Val Loss)")

 ✅ Saved Best Model (based on Val Loss)


In [ ]:
model_best = MultiTaskViT5(n_sent, n_topic) # n_sent, n_topic lấy từ cell train trước đó
model_best.to(Config.DEVICE)

# Load weight đã train
try:
    model_best.load_state_dict(torch.load("vit5_last.pth"))
    print("✅ Đã load weight từ 'vit5_last.pth'")
except Exception as e:
    print(f"⚠️ Không tìm thấy file weight, dùng model hiện tại: {e}")

def evaluate_test(model, dataloader, device):
    model.eval()
    true_sents, pred_sents = [], []
    true_topics, pred_topics = [], []
    
    print("Đang chạy đánh giá trên tập Test...")
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            
            labels_s = batch['sentiment_label'].cpu().numpy()
            labels_t = batch['topic_label'].cpu().numpy()
            
            # Forward
            logits_s, logits_t = model(input_ids, mask)
            
            # Predict
            preds_s = torch.argmax(logits_s, dim=1).cpu().numpy()
            preds_t = torch.argmax(logits_t, dim=1).cpu().numpy()
            
            true_sents.extend(labels_s)
            pred_sents.extend(preds_s)
            true_topics.extend(labels_t)
            pred_topics.extend(preds_t)
            
    return true_sents, pred_sents, true_topics, pred_topics

y_true_s, y_pred_s, y_true_t, y_pred_t = evaluate_test(model_best, test_loader, Config.DEVICE)

# --- SENTIMENT REPORT ---
print("\n" + "="*40)
print("KẾT QUẢ: CẢM XÚC (SENTIMENT)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_s, y_pred_s):.4f}\n")
s_names = [ID2SENTIMENT[i] for i in sorted(ID2SENTIMENT.keys()) if i < n_sent]
print(classification_report(y_true_s, y_pred_s, target_names=s_names, zero_division=0))

# --- TOPIC REPORT ---
print("\n" + "="*40)
print("KẾT QUẢ: CHỦ ĐỀ (TOPIC)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_t, y_pred_t):.4f}\n")
t_names = [ID2TOPIC[i] for i in sorted(ID2TOPIC.keys()) if i < n_topic]
print(classification_report(y_true_t, y_pred_t, target_names=t_names, zero_division=0))


def predict_single(text, model, tokenizer, device):
    model.eval()
    # 1. Tách từ
    text_seg = segment_text(text)
    # 2. Tokenize
    encoding = tokenizer.encode_plus(
        text_seg,
        max_length=Config.MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        l_s, l_t = model(input_ids, mask)
        p_s = torch.argmax(l_s, dim=1).item()
        p_t = torch.argmax(l_t, dim=1).item()
        
    return ID2SENTIMENT.get(p_s, "Unknown"), ID2TOPIC.get(p_t, "Unknown")

print("\n--- DEMO PREDICTION ---")
sample_txt = "Trường mình cơ sở vật chất tốt nhưng học phí hơi cao so với mặt bằng chung."
print(f"Input: {sample_txt}")
s, t = predict_single(sample_txt, model_best, tokenizer, Config.DEVICE)
print(f"Sentiment: {s}")
print(f"Topic: {t}")

Loading VietAI/vit5-base...
✅ Đã load weight từ 'vit5_last.pth'
Đang chạy đánh giá trên tập Test...


  0%|          | 0/827 [00:00<?, ?it/s]


KẾT QUẢ: CẢM XÚC (SENTIMENT)
Accuracy: 0.8096

              precision    recall  f1-score   support

     Neutral       0.87      0.88      0.87      4558
    Positive       0.67      0.63      0.65       834
    Negative       0.64      0.65      0.64      1046
       Toxic       0.92      0.78      0.84       175

    accuracy                           0.81      6613
   macro avg       0.78      0.73      0.75      6613
weighted avg       0.81      0.81      0.81      6613


KẾT QUẢ: CHỦ ĐỀ (TOPIC)
Accuracy: 0.7599

                    precision    recall  f1-score   support

              Spam       0.51      0.45      0.48        84
              News       0.42      0.64      0.50       183
          Academic       0.82      0.80      0.81      2104
             Other       0.83      0.86      0.84      2882
           Service       0.60      0.64      0.62       474
Jobs & Recruitment       0.61      0.62      0.61       163
  Personal Affairs       0.56      0.41      0.47    